In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask.dataframe as dd
import pickle
import os

In [ ]:
# data_dir = "sameBlock"
# data_dir = "sameSuperblock"
data_dir = "sameBlockWithHigherOrderStats"

In [ ]:
# —Later— to load:
if data_dir != "sameBlockWithHigherOrderStats":
    with open(f'./{data_dir}/estimators.pkl', 'rb') as f:
        estimators = pickle.load(f)
else:
    with open(f'./{data_dir}/standard_disjoint_estimators.pkl', 'rb') as f:
        standard_disjoint_estimators = pickle.load(f)
    with open(f'./{data_dir}/standard_overlapping_estimators.pkl', 'rb') as f:
        standard_overlapping_estimators = pickle.load(f)
    with open(f'./{data_dir}/standard_sampled_estimators.pkl', 'rb') as f:
        standard_sampled_estimators = pickle.load(f)
    with open(f'./{data_dir}/bs_disjoint_estimators.pkl', 'rb') as f:
        bs_disjoint_estimators = pickle.load(f)
    with open(f'./{data_dir}/bs_overlapping_estimators.pkl', 'rb') as f:
        bs_overlapping_estimators = pickle.load(f)
    with open(f'./{data_dir}/bs_sampled_estimators.pkl', 'rb') as f:
        bs_sampled_estimators = pickle.load(f)

        # Bundle all dicts into one object
    estimators = {
        'means': {
            "standard_disjoint_estimator": standard_disjoint_estimators["standard_disjoint_estimator"],
            "bs_disjoint_estimator": bs_disjoint_estimators["bs_disjoint_estimator"],
            "standard_overlapping_estimator": standard_overlapping_estimators["standard_overlapping_estimator"],
            "bs_overlapping_estimator": bs_overlapping_estimators["bs_overlapping_estimator"],
            "standard_sampled_estimator": standard_sampled_estimators["standard_sampled_estimator"],
            "bs_sampled_estimator": bs_sampled_estimators["bs_sampled_estimator"],
        },
        'stds': {
            "standard_disjoint_std_estimator": standard_disjoint_estimators["standard_disjoint_std_estimator"],
            "bs_disjoint_std_estimator": bs_disjoint_estimators["bs_disjoint_std_estimator"],
            "standard_overlapping_std_estimator": standard_overlapping_estimators["standard_overlapping_std_estimator"],
            "bs_overlapping_std_estimator": bs_overlapping_estimators["bs_overlapping_std_estimator"],
            "standard_sampled_std_estimator": standard_sampled_estimators["standard_sampled_std_estimator"],
            "bs_sampled_std_estimator": bs_sampled_estimators["bs_sampled_std_estimator"]
        },
        'skews': {
            "standard_disjoint_skew_estimator": standard_disjoint_estimators["standard_disjoint_skew_estimator"],
            "bs_disjoint_skew_estimator": bs_disjoint_estimators["bs_disjoint_skew_estimator"],
            "standard_overlapping_skew_estimator": standard_overlapping_estimators["standard_overlapping_skew_estimator"],
            "bs_overlapping_skew_estimator": bs_overlapping_estimators["bs_overlapping_skew_estimator"],
            "standard_sampled_skew_estimator": standard_sampled_estimators["standard_sampled_skew_estimator"],
            "bs_sampled_skew_estimator": bs_sampled_estimators["bs_sampled_skew_estimator"]
        },
        'kurts': {
            "standard_disjoint_kurt_estimator": standard_disjoint_estimators["standard_disjoint_kurt_estimator"],
            "bs_disjoint_kurt_estimator": bs_disjoint_estimators["bs_disjoint_kurt_estimator"],
            "standard_overlapping_kurt_estimator": standard_overlapping_estimators["standard_overlapping_kurt_estimator"],
            "bs_overlapping_kurt_estimator": bs_overlapping_estimators["bs_overlapping_kurt_estimator"],
            "standard_sampled_kurt_estimator": standard_sampled_estimators["standard_sampled_kurt_estimator"],
            "bs_sampled_kurt_estimator": bs_sampled_estimators["bs_sampled_kurt_estimator"]
        }
    }

In [ ]:

# —Later— to reload:
with open(f'./{data_dir}/all_maximas_summary.pkl', 'rb') as f:
    summary_stats = pickle.load(f)


In [ ]:
summary_stats

In [ ]:
# Create a DataFrame to hold the summary statistics
summary_data = []

# Extract mean values from each calculation method for each series
for calc_method, series_dict in summary_stats.items():
    for series_name, stats in series_dict.items():
        # For overlapping, we have multiple columns (SBM, CBM_k2, CBM_k5)
        if calc_method == 'overlapping':
            for col in stats.columns:
                mean_value = stats[col]['mean']
                summary_data.append({
                    'Series': series_name,
                    'Calculation Method': f"{calc_method}_{col}",
                    'Mean Value': mean_value
                })
        else:
            # For disjoint and sampled, we just have SBM
            mean_value = stats['SBM']['mean']
            summary_data.append({
                'Series': series_name,
                'Calculation Method': f"{calc_method}_SBM",
                'Mean Value': mean_value
            })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_data)

# Create pivot table for easier plotting
plot_df = summary_df.pivot(index='Series', columns='Calculation Method', values='Mean Value')

# Plot grouped bar chart for means
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(plot_df.index))  # Series types
width = 0.8 / len(plot_df.columns)  # Width of each bar

for i, col in enumerate(plot_df.columns):
    offset = i * width - (len(plot_df.columns) - 1) * width / 2
    ax.bar(x + offset, plot_df[col], width, label=col)

# Add labels, title, and legend
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=45)
ax.set_ylabel('Mean Value')
ax.set_title('Summary Statistics: Mean Values by Series and Calculation Method')
ax.legend(title='Calculation Method', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Create a DataFrame to hold the standard deviation statistics
std_summary_data = []

# Extract std values from each calculation method for each series
for calc_method, series_dict in summary_stats.items():
    for series_name, stats in series_dict.items():
        # For overlapping, we have multiple columns (SBM, CBM_k2, CBM_k5)
        if calc_method == 'overlapping':
            for col in stats.columns:
                std_value = stats[col]['std']
                std_summary_data.append({
                    'Series': series_name,
                    'Calculation Method': f"{calc_method}_{col}",
                    'Std Value': std_value
                })
        else:
            # For disjoint and sampled, we just have SBM
            std_value = stats['SBM']['std']
            std_summary_data.append({
                'Series': series_name,
                'Calculation Method': f"{calc_method}_SBM",
                'Std Value': std_value
            })

# Convert to DataFrame
std_summary_df = pd.DataFrame(std_summary_data)

# Create pivot table for easier plotting
std_plot_df = std_summary_df.pivot(index='Series', columns='Calculation Method', values='Std Value')

# Plot grouped bar chart for standard deviations
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(std_plot_df.index))  # Series types
width = 0.8 / len(std_plot_df.columns)  # Width of each bar

for i, col in enumerate(std_plot_df.columns):
    offset = i * width - (len(std_plot_df.columns) - 1) * width / 2
    ax.bar(x + offset, std_plot_df[col], width, label=col)

# Add labels, title, and legend
ax.set_xticks(x)
ax.set_xticklabels(std_plot_df.index, rotation=45)
ax.set_ylabel('Standard Deviation')
ax.set_title('Summary Statistics: Standard Deviations by Series and Calculation Method')
ax.legend(title='Calculation Method', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
suffix_map = {
    'means': '_mean',
    'stds': '_std',
    'skews': '_skew',
    'kurts': '_kurt',
}

for metric, suffix in suffix_map.items():
    if metric not in estimators:
        continue

    for estimator_name, metric_series in estimators[metric].items():
        for series_name in metric_series.keys():
            estimators[metric][estimator_name][series_name] = metric_series[series_name].rename(
                lambda x, suffix=suffix: x.replace(suffix, ''), axis=1
            )

In [ ]:
# Create figures for each metric family. Means/stds use true-value references;
# skews/kurts are plotted in no-reference mode because summary_stats does not include them.
figures = {}
axes = {}

plot_configs = [
    ('means', 'Distributions of Means', (2.5, 4.5), 'mean'),
    ('stds', 'Distributions of Standard Deviations', (0, 1), 'std'),
    ('skews', 'Distributions of Skewness Estimates', None, None),
    ('kurts', 'Distributions of Kurtosis Estimates', None, None),
]

for metric, title, x_limits, reference_stat in plot_configs:
    if metric not in estimators or len(estimators[metric]) == 0:
        continue

    combined_data = estimators[metric]
    series_names = sorted(combined_data[list(combined_data.keys())[0]].keys())

    fig, axs = plt.subplots(len(series_names), 1, figsize=(12, 8), constrained_layout=True)
    if len(series_names) == 1:
        axs = [axs]

    figures[metric] = fig
    axes[metric] = axs

    for estimator_name in combined_data.keys():
        data = combined_data[estimator_name]

        for col_idx, series_name in enumerate(series_names):
            ax = axs[col_idx]

            if x_limits is not None:
                ax.set_xlim(*x_limits)

            true_stats = None
            if reference_stat is not None:
                stat_group = estimator_name.split('_')[1]
                true_stats = summary_stats.get(stat_group, {}).get(series_name, {})

            for col in data[series_name].columns:
                values = data[series_name][col].dropna().values

                if x_limits is not None:
                    bins = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 121)
                else:
                    bins = 60

                ax.hist(values, bins=bins, alpha=0.1, label=f'{col} {estimator_name}')

                if true_stats is not None and col in true_stats:
                    true_value = true_stats[col].get(reference_stat, np.nan)
                    if pd.notna(true_value):
                        ax.axvline(
                            true_value,
                            color='r' if metric == 'means' else 'b',
                            linestyle='--',
                            linewidth=2,
                            label=None,
                        )

            ax.set_title(f'{series_name}')
            ax.set_xlabel('Estimate')
            ax.set_ylabel('Count')

            if col_idx == len(series_names) - 1:
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    fig.suptitle(title, fontsize=16)
    plt.show()

In [ ]:
# Assuming we already have:
# 1) estimators['means'], estimators['stds'], estimators['skews'], estimators['kurts']
# 2) summary_stats with true mean/std references (no true skew/kurt references)

# Create a function to process a specific estimator type.
# bias/mse require a true reference; se/iqr are always computed from estimator samples.
def compute_metrics(est_type, combined_df, reference_metric=None):
    all_series_names = sorted({
        series_name
        for current_est_type in combined_df.keys()
        for series_name in combined_df[current_est_type].keys()
    })

    first_series = next(iter(combined_df[est_type].keys()))
    cols = combined_df[est_type][first_series].columns

    bias = pd.DataFrame(index=all_series_names, columns=cols, dtype=float)
    se = pd.DataFrame(index=all_series_names, columns=cols, dtype=float)
    mse = pd.DataFrame(index=all_series_names, columns=cols, dtype=float)
    iqr = pd.DataFrame(index=all_series_names, columns=cols, dtype=float)

    for series_name in all_series_names:
        if series_name not in combined_df[est_type]:
            continue

        for col in cols:
            est = combined_df[est_type][series_name][col].dropna().values
            if est.size == 0:
                bias.at[series_name, col] = np.nan
                se.at[series_name, col] = np.nan
                mse.at[series_name, col] = np.nan
                iqr.at[series_name, col] = np.nan
                continue

            se.at[series_name, col] = est.std(ddof=0)
            iqr.at[series_name, col] = np.percentile(est, 75) - np.percentile(est, 25)

            if reference_metric is None:
                bias.at[series_name, col] = np.nan
                mse.at[series_name, col] = np.nan
                continue

            stat_group = est_type.split('_')[1]
            true = (
                summary_stats
                .get(stat_group, {})
                .get(series_name, {})
                .get(col, {})
                .get(reference_metric, np.nan)
            )

            if pd.notna(true):
                bias.at[series_name, col] = est.mean() - true
                mse.at[series_name, col] = np.mean((est - true) ** 2)
            else:
                bias.at[series_name, col] = np.nan
                mse.at[series_name, col] = np.nan

    return {
        'bias': bias,
        'se': se,
        'mse': mse,
        'iqr': iqr,
    }

analysis_specs = [
    ('means', estimators.get('means', {}), 'mean'),
    ('stds', estimators.get('stds', {}), 'std'),
    ('skews', estimators.get('skews', {}), None),
    ('kurts', estimators.get('kurts', {}), None),
]

results_by_metric = {}
for metric_name, metric_estimators, reference_metric in analysis_specs:
    if len(metric_estimators) == 0:
        continue

    results_by_metric[metric_name] = {}
    estimator_types = list(metric_estimators.keys())

    for est_type in estimator_types:
        results_by_metric[metric_name][est_type] = compute_metrics(
            est_type, metric_estimators, reference_metric
        )

        print(f"\n=== Metrics for {metric_name} :: {est_type} ===")
        if reference_metric is None:
            print("No true-reference stats available; bias and MSE are reported as NaN.")

        print("Bias estimates:\n", results_by_metric[metric_name][est_type]['bias'])
        print("\nSE estimates:\n", results_by_metric[metric_name][est_type]['se'])
        print("\nMSE estimates:\n", results_by_metric[metric_name][est_type]['mse'])
        print("\nIQR estimates:\n", results_by_metric[metric_name][est_type]['iqr'])

# Backward-compatible aliases for downstream cells.
results_means = results_by_metric.get('means', {})
results_stds = results_by_metric.get('stds', {})
results_skews = results_by_metric.get('skews', {})
results_kurts = results_by_metric.get('kurts', {})

In [ ]:
metric_labels = {
    'bias': 'Bias',
    'se': 'SE',
    'mse': 'MSE',
    'iqr': 'IQR',
}
plot_order = ['bias', 'se', 'mse', 'iqr']

for metric_name, metric_results in results_by_metric.items():
    if len(metric_results) == 0:
        continue

    estimator_types = list(metric_results.keys())

    available_metrics = []
    for plot_metric in plot_order:
        has_data = any(
            not metric_results[est_type][plot_metric].isna().all().all()
            for est_type in estimator_types
        )
        if has_data:
            available_metrics.append(plot_metric)

    if len(available_metrics) == 0:
        print(f"Skipping {metric_name}: no plottable metrics.")
        continue

    fig, axs = plt.subplots(
        len(available_metrics),
        len(estimator_types),
        figsize=(15, 5 * len(available_metrics)),
        layout='tight',
        sharey='row',
    )

    if len(available_metrics) == 1 and len(estimator_types) == 1:
        axs = np.array([[axs]])
    elif len(available_metrics) == 1:
        axs = axs.reshape(1, -1)
    elif len(estimator_types) == 1:
        axs = axs.reshape(-1, 1)

    for col_idx, est_type in enumerate(estimator_types):
        for row_idx, plot_metric in enumerate(available_metrics):
            ax = axs[row_idx, col_idx]

            df = metric_results[est_type][plot_metric].T
            x = np.arange(len(df.index))
            width = 0.7 / len(df.columns)

            for j, col in enumerate(df.columns):
                offset = width * j - (len(df.columns) - 1) * width / 2
                ax.bar(x + offset, df[col], width, alpha=0.5, label=f'{col}')

            ax.set_title(est_type.replace('_estimator', ''))
            ax.set_xticks(x)
            ax.set_xticklabels(df.index, rotation=45)

            if row_idx == 0 and col_idx == len(estimator_types) - 1:
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

            if col_idx == 0:
                ax.set_ylabel(f"{metric_labels[plot_metric]} of {metric_name[:-1].capitalize()} Estimates")

    fig.suptitle(f"{metric_name.capitalize()} Metrics", fontsize=16, y=1.02)
    plt.show()